In [42]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report 
from sklearn.metrics import confusion_matrix
from sklearn.metrics import pairwise_distances

df=pd.read_csv("breast-cancer.csv")
print("first 5 rows")
print(df.head())
print("\n Dataset shape:")
print(df.shape)
print("\ncolumns names:")
print(df.columns)

if'id' in df.columns:
    df=df.drop('id',axis=1)

if 'Unnamed: 32' in df.columns:
    df=df.drop('Unnamed: 32',axis=1)
if 'diagnosis' in df.columns:
    df['diagnosis']=df['diagnosis'].map({'M':1,'B':0})
X= df.drop('diagnosis',axis=1)
y=df['diagnosis']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

print("\nTraning samples:",X_train.shape[0])
print("Testing samples:",X_test.shape[0])
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

k_values =[1,3,5,7,9,11,13,15]
accuracies=[] 
print("\nAccuracy for different values of k:")
print("------------------------------------")
for k in k_values:
    knn=KNeighborsClassifier(n_neighbors=k,metric='euclidean')
    knn.fit(X_train_scaled,y_train)
    y_pred=knn.predict(X_test_scaled)
    accuracy=accuracy_score(y_test,y_pred)
    accuracies.append(accuracy)
    print("k =",k,"Accuracy =",round(accuracy,4))

best_index = np.argmax(accuracies)
best_k = k_values[best_index]
best_accuracy = accuracies[best_index]
print("\nBest value of k:", best_k)
print("Best Accuracy:",round(best_accuracy,4))


knn_model = KNeighborsClassifier(n_neighbors=best_k,metric='euclidean')
knn_model.fit(X_train_scaled,y_train)
y_pred = knn_model.predict(X_test_scaled)
print("\nFinal Moodel Performance")
print("Accuracy:", round(accuracy_score(y_test,y_pred),4))
print("\n Confusion Matrix:")
print(confusion_matrix(y_test,y_pred))
print("\nClassification Report:")
print(classification_report(y_test,y_pred))



first 5 rows
         id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  radius_worst  texture_worst  perimeter_wo

In [43]:
selected_index=0
selected_observation=X_test_scaled[selected_index]

print("\nSelected Observation:")
print(selected_observation)
distances=pairwise_distances(
    X_train_scaled,
    selected_observation.reshape(1,-1),
    metric='euclidean'
).flatten()
nearest_indices=np.argsort(distances)[:best_k]

print("\nNearest neighbors:")
print("---------------------")
for i, index in enumerate(nearest_indices):

    print(
        "Neighbor",i+1,
        "| Training index:",index,
        "| Distance:",round(distances[index],4),
        "| Class:",y_train.iloc[index]
    )




Selected Observation:
[-0.77089916 -2.00602473 -0.76451652 -0.71183971 -0.1583148  -0.68680034
 -0.66248612 -0.57475415 -0.53582061 -0.22760708 -0.93280815 -1.37244026
 -0.85278289 -0.6368401  -0.2999339  -0.56993036 -0.53250895 -0.83258923
 -0.8416237  -0.59337276 -0.72135781 -1.63595077 -0.70654903 -0.65337009
  0.96851866 -0.10446758 -0.30668651 -0.38805702  0.16254727  0.06095804]

Nearest neighbors:
---------------------
Neighbor 1 | Training index: 236 | Distance: 1.5954 | Class: 0
Neighbor 2 | Training index: 428 | Distance: 1.8849 | Class: 0
Neighbor 3 | Training index: 164 | Distance: 1.9156 | Class: 0
Neighbor 4 | Training index: 34 | Distance: 1.9573 | Class: 0
Neighbor 5 | Training index: 98 | Distance: 2.0695 | Class: 0


In [44]:
nearest_labels=y_train.iloc[nearest_indices].values
print("\nClasses of nearest neighbors:")
print(nearest_labels)
unique_classes,counts=np.unique(
    nearest_labels,
    return_counts=True
)
print("\nVoting:")
for class_value,count in zip(unique_classes,counts):
    print("Class:",class_value,":",count,"votes")

majority_class=unique_classes[np.argmax(counts)]
print("\nPredicted class using majority voting:",majority_class)
print("\Actual class :",y_test.iloc[selected_index])



Classes of nearest neighbors:
[0 0 0 0 0]

Voting:
Class: 0 : 5 votes

Predicted class using majority voting: 0
\Actual class : 0


<>:14: SyntaxWarning: invalid escape sequence '\A'
<>:14: SyntaxWarning: invalid escape sequence '\A'
C:\Users\Vaish\AppData\Local\Temp\ipykernel_8584\12475208.py:14: SyntaxWarning: invalid escape sequence '\A'
  print("\Actual class :",y_test.iloc[selected_index])


In [45]:
print("\nEffect of different distance metrics:")
print("---------------------------")
metrics =['euclidean','manhattan']
for metric in metrics:
    model=KNeighborsClassifier(n_neighbors=best_k,metric=metric)
    model.fit(X_train_scaled,y_train)
    prediction=model.predict(X_test_scaled)
    accuracy=accuracy_score(y_test,prediction)
    print(metric.capitalize(),"Accuracy =",round(accuracy,4))



Effect of different distance metrics:
---------------------------
Euclidean Accuracy = 0.9561
Manhattan Accuracy = 0.9561


In [41]:
new_observation=X_test.iloc[[0]]
print("\nnew observation")
print(new_observation)

new_observation_scaled=scaler.transform(
    new_observation
)

new_prediction=knn_model.predict(
    new_observation_scaled
)
print("\nPredicted class:",new_prediction[0])

if new_prediction[0]==1:
    print("Prediction: Malignant (M)")
else:
    print("Prediction: Benign (B)")


new observation
     radius_mean  texture_mean  perimeter_mean  area_mean  smoothness_mean  \
120        11.41         10.82           73.34      403.3          0.09373   

     compactness_mean  concavity_mean  concave points_mean  symmetry_mean  \
120           0.06685         0.03512              0.02623         0.1667   

     fractal_dimension_mean  ...  radius_worst  texture_worst  \
120                 0.06113  ...         12.82          15.97   

     perimeter_worst  area_worst  smoothness_worst  compactness_worst  \
120            83.74       510.5            0.1548              0.239   

     concavity_worst  concave points_worst  symmetry_worst  \
120           0.2102               0.08958          0.3016   

     fractal_dimension_worst  
120                  0.08523  

[1 rows x 30 columns]

Predicted class: 0
Prediction: Benign (B)
